In [1]:
import sqlite3
import pandas as pd
import json
import os
from dotenv import load_dotenv

load_dotenv()
DB_NAME = os.getenv('DB_NAME', 'jobscope.db')
conn = sqlite3.connect(DB_NAME)

In [2]:
# Load both tables into DataFrames
raw_df = pd.read_sql('SELECT * FROM raw_jobs', conn)
clean_df = pd.read_sql('SELECT * FROM clean_jobs', conn)

print(f'Raw jobs: {len(raw_df)}')
print(f'Clean jobs: {len(clean_df)}')

Raw jobs: 2765
Clean jobs: 2064


In [ ]:
# Quick look at clean data
clean_df.head()

In [ ]:
# Column types and nulls
clean_df.info()

In [ ]:
# Parse the extracted_skills JSON column into a Python list
clean_df['skills_list'] = clean_df['extracted_skills'].apply(json.loads)
clean_df['num_skills'] = clean_df['skills_list'].apply(len)

clean_df[['title_original', 'role_category', 'seniority', 'location_region', 'salary_mid', 'num_skills']].head(10)

In [ ]:
# Quick value counts
print('--- Role Categories ---')
print(clean_df['role_category'].value_counts())
print('\n--- Seniority ---')
print(clean_df['seniority'].value_counts())
print('\n--- Top 10 Regions ---')
print(clean_df['location_region'].value_counts().head(10))

In [ ]:
# Salary overview (only real salaries)
salary_df = clean_df[clean_df['has_real_salary'] == 1].copy()
print(f'Jobs with real salary: {len(salary_df)}')
salary_df['salary_mid'].describe()

In [ ]:
# Don't forget to close when done
# conn.close()

In [3]:
# 1. Skipped jobs — what titles are we missing?
print("=== TOP 30 SKIPPED TITLES ===")
skipped = pd.read_sql("""
    SELECT r.title, r.search_term, COUNT(*) as cnt 
    FROM raw_jobs r
    LEFT JOIN clean_jobs c ON c.raw_job_id = r.id
    WHERE c.id IS NULL
    GROUP BY r.title
    ORDER BY cnt DESC
    LIMIT 30
""", conn)
print(skipped.to_string(index=False))

=== TOP 30 SKIPPED TITLES ===
                                             title                   search_term  cnt
                              Data Science Trainee                data scientist   95
                                   Product Analyst                  data analyst   34
                                   Pricing Analyst                  data analyst   20
                                  Business Analyst business intelligence analyst   16
                                      FP&A Analyst                  data analyst   12
                    Trainee Field Service Engineer     machine learning engineer    9
                                   Finance Analyst                  data analyst    9
                              Head of Data Science                data scientist    8
                          Senior Software Engineer                  LLM engineer    7
                               Head of Engineering                  LLM engineer    7
             Technical L

In [4]:
# 2. "Other UK" locations — what are the actual values?
print("\n=== TOP 30 'OTHER UK' LOCATIONS ===")
other_locs = pd.read_sql("""
    SELECT location_raw, COUNT(*) as cnt
    FROM clean_jobs 
    WHERE location_region = 'Other UK'
    GROUP BY location_raw
    ORDER BY cnt DESC
    LIMIT 30
""", conn)
print(other_locs.to_string(index=False))


=== TOP 30 'OTHER UK' LOCATIONS ===
                    location_raw  cnt
                              UK  156
                  United Kingdom   13
Newcastle Upon Tyne, Tyne & Wear   11
                       Sheffield    9
     Cheltenham, Gloucestershire    8
             Telford, Shropshire    7
     Nottingham, Nottinghamshire    7
             Newcastle Upon Tyne    6
                   Milton Keynes    6
        Stevenage, Hertfordshire    5
                          SW21RW    5
                          PR12RL    5
  Milton Keynes, Buckinghamshire    5
                          LU12BQ    5
            Warrington, Cheshire    4
                           W52BJ    4
          Southampton, Hampshire    4
                         SO147LY    4
                          PL13BJ    4
                           N12UD    4
                          MK93HQ    4
                       Leicester    4
     Gloucester, Gloucestershire    4
                   Exeter, Devon    4
             

In [5]:
# 3. Skill extraction check — sample a description that should have Python
print("\n=== SAMPLE DESCRIPTION (first Data Scientist job) ===")
sample = pd.read_sql("""
    SELECT description_clean, extracted_skills
    FROM clean_jobs 
    WHERE role_category = 'Data Scientist'
    LIMIT 1
""", conn)
print(sample['description_clean'].iloc[0][:500])
print("\nExtracted:", sample['extracted_skills'].iloc[0])



=== SAMPLE DESCRIPTION (first Data Scientist job) ===
are you ready to start a new career in data analysis? the demand for data analysts has grown by 20% annually, with experienced professionals earning salaries upwards of £58,000. in today’s digital world, data is critical to business decision-making, making the role of a data analyst indispensable. as skills shortages continue to grow, the demand for qualified entry-level professionals is on the rise. with our data analytics career programme we will provide you with: 8 training modules: excel, s…

Extracted: ["excel"]


In [6]:
# Run this to check if Reed descriptions are longer
import sqlite3, os
from dotenv import load_dotenv
load_dotenv()
conn = sqlite3.connect(os.getenv('DB_NAME', 'jobscope.db'))

for source in ['adzuna', 'reed']:
    rows = conn.execute(
        "SELECT AVG(LENGTH(description)), MAX(LENGTH(description)), MIN(LENGTH(description)) FROM raw_jobs WHERE source = ?",
        (source,)
    ).fetchone()
    print(f"{source}: avg={rows[0]:.0f}, max={rows[1]}, min={rows[2]}")

conn.close()

adzuna: avg=499, max=500, min=105
reed: avg=453, max=453, min=255
